# Tutorial 5: DoubleZernikes

`DoubleZernikes` is a 2D Zernike basis for **field-dependent wavefronts**.  The
coefficient array has axes for both **field** Zernike index *k* and **pupil**
Zernike index *j*.

This compact representation is used to describe how the wavefront changes across
the field of view with a small number of parameters.

## Key concepts

| | Field axis (*k*) | Pupil axis (*j*) |
|---|---|---|
| **Aperture** | `field_outer`, `field_inner` | `pupil_outer`, `pupil_inner` |
| **Max index** | `kmax` | `jmax` |

- Created from `Zernikes.double()` (least-squares fit) or constructed directly.
- Evaluated at specific field points via `.single(field)` → `Zernikes`.
- Frame rotation rotates **both** field and pupil coordinates simultaneously.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import Angle
import galsim

from StarSharp import FieldCoords, Zernikes, DoubleZernikes

%matplotlib inline

## 1. Construction from Zernikes

The typical way to create a `DoubleZernikes` is from a `Zernikes` object with
many field points.  The `.double()` method performs a least-squares fit of the
field-dependent coefficients to the field Zernike basis.

In [ ]:
rtp = Angle(30, unit=u.deg)
R_outer = 4.18 * u.m
R_inner = 2.56 * u.m
field_outer = 1.75 * u.deg

# Create a dense field grid for a good fit
n_field = 50
rng = np.random.default_rng(42)
x = rng.uniform(-1.5, 1.5, n_field)
y = rng.uniform(-1.5, 1.5, n_field)
field = FieldCoords(
    x=x * u.deg,
    y=y * u.deg,
    frame="ocs",
    rtp=rtp,
)

# Simulated field-dependent Zernikes (jmax=22)
jmax = 22
coefs = ((x*y)[:, None] + rng.normal(0.0, 0.5, (n_field, jmax + 1))) * u.um
coefs[:, :4] = 0 * u.um  # zero piston/tip/tilt

zk = Zernikes(
    coefs=coefs, field=field,
    R_outer=R_outer, R_inner=R_inner,
    wavelength=622 * u.nm, frame="ocs", rtp=rtp,
)

# Fit to DoubleZernikes
dz = zk.double(kmax=6, field_outer=field_outer)

print(f"Type: {type(dz).__name__}")
print(f"kmax (field): {dz.kmax}")
print(f"jmax (pupil): {dz.jmax}")
print(f"coefs shape:  {dz.coefs.shape}  →  (kmax+1, jmax+1)")
print(f"eps (pupil):  {dz.eps:.3f}")

## 2. Direct construction

You can also construct `DoubleZernikes` directly from a coefficient array.

In [ ]:
# Build a DoubleZernikes directly: kmax=3, jmax=10
kmax_direct = 3
jmax_direct = 10
coefs_direct = np.zeros((kmax_direct + 1, jmax_direct + 1)) * u.um

# Set a few non-zero coefficients:
#   k=1 (field-constant) for j=5 (astigmatism) — uniform astigmatism
coefs_direct[1, 5] = 0.03 * u.um
#   k=2 (field-linear) for j=6 (astigmatism) — linear astigmatism
coefs_direct[2, 6] = 0.01 * u.um
#   k=1 (field-constant) for j=4 (defocus) — uniform defocus
coefs_direct[1, 4] = 0.05 * u.um

dz_direct = DoubleZernikes(
    coefs=coefs_direct,
    field_outer=1.75 * u.deg,
    field_inner=0.0 * u.deg,
    pupil_outer=R_outer,
    pupil_inner=R_inner,
    frame="ocs",
    rtp=rtp,
)

print(f"kmax: {dz_direct.kmax}, jmax: {dz_direct.jmax}")
print(f"batch_shape: {dz_direct.batch_shape}")

## 3. Frame conversions

Frame rotation for `DoubleZernikes` rotates **both** the field Zernike indices
(*k*) and the pupil Zernike indices (*j*) simultaneously:

$$C'_{k'j'} = \sum_{k,j} R^{\text{field}}_{k'k} \; C_{kj} \; R^{\text{pupil}}_{jj'}$$

In [ ]:
dz_ccs = dz.ccs
print(f"Frame: {dz.frame} → {dz_ccs.frame}")
print(f"Shape preserved: {dz.coefs.shape} → {dz_ccs.coefs.shape}")

In [ ]:
# Roundtrip: OCS → CCS → OCS
dz_rt = dz.ccs.ocs
print(f"Roundtrip close: {np.allclose(dz.coefs, dz_rt.coefs)}")
print(f"Max error: {np.max(np.abs(dz.coefs - dz_rt.coefs)):.2e}")

### Visualizing the coefficient matrix

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for ax, (label, d) in zip([ax1, ax2], [("OCS", dz_direct), ("CCS", dz_direct.ccs)]):
    im = ax.imshow(
        d.coefs.to_value(u.um), aspect="auto",
        cmap="RdBu", origin="lower",
        vmin=-0.06, vmax=0.06,
    )
    ax.set_xlabel("Pupil Noll j")
    ax.set_ylabel("Field Noll k")
    ax.set_title(f"DoubleZernike coefficients ({label})")
    plt.colorbar(im, ax=ax, label="[µm]")

fig.suptitle(f"Frame rotation mixes both field and pupil indices (rtp={rtp.deg:.0f}°)", fontsize=13)

## 4. Bridge: DoubleZernikes → Zernikes via `.single(field)`

`.single(field)` evaluates the field Zernike basis at specified field
coordinates and returns a `Zernikes` object — the inverse of `.double()`.

In [ ]:
# Evaluate at a few specific field points
eval_field = FieldCoords(
    x=np.array([0.0, 0.5, -1.0]) * u.deg,
    y=np.array([0.0, 0.5, 0.8]) * u.deg,
    frame="ocs",
    rtp=rtp,
)

zk_from_dz = dz.single(eval_field)

print(f"Type: {type(zk_from_dz).__name__}")
print(f"nfield: {zk_from_dz.nfield}")
print(f"jmax:   {zk_from_dz.jmax}")
print(f"frame:  {zk_from_dz.frame}")

In [ ]:
# The center of the field (0, 0) should match the k=1 coefficients
center_field = FieldCoords(
    x=np.array([0.0]) * u.deg,
    y=np.array([0.0]) * u.deg,
    frame="ocs",
    rtp=rtp,
)
zk_center = dz_direct.single(center_field)

# At field center, only k=1 contributes (k≥2 field Zernikes vanish at origin)
# So the pupil coefficients should just be dz_direct.coefs[1, :]
print(f"DoubleZernike k=1 row:     {dz_direct.coefs[1, :6]}")
print(f"Zernikes at field center:  {zk_center.coefs[0, :6]}")
print(f"Match: {np.allclose(dz_direct.coefs[1, :], zk_center.coefs[0, :])}")

## 5. Roundtrip: Zernikes → DoubleZernikes → Zernikes

Starting from Zernikes at many field points, fitting to DoubleZernikes, then
evaluating back at the same points should approximately recover the original
(limited by the field-Zernike truncation at `kmax`).

In [ ]:
# Roundtrip using the dense-field Zernikes from Section 1
zk_roundtrip = dz.single(field)

residual_per_field = np.sqrt(np.mean((zk.coefs - zk_roundtrip.coefs).to_value(u.um)**2, axis=-1))

fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(10, 5), constrained_layout=True)
axs[0].scatter(
    field.x.to_value(u.deg), field.y.to_value(u.deg),
    c=zk.coefs[:, 4].to_value(u.um), cmap="viridis", s=100, edgecolor="k",
)
axs[1].scatter(
    field.x.to_value(u.deg), field.y.to_value(u.deg),
    c=zk_roundtrip.coefs[:, 4].to_value(u.um), cmap="viridis", s=100, edgecolor="k",
)
axs[0].set_title("Original Zernike coefficients (j=4)")
axs[1].set_title("Recovered Zernike coefficients (j=4)")
for ax in axs:
    ax.set_xlabel("Field x [deg]")
    ax.set_ylabel("Field y [deg]")
plt.show()

## 6. GalSim interop

In [ ]:
gdz = dz.to_galsim()
print(f"Type: {type(gdz)}")
print(f"GalSim DoubleZernike coefs shape: {np.array(gdz.coef).shape}")

## 7. Indexing and batch dimensions

Indexing into a `DoubleZernikes` selects along the leading batch dimension
(not the k or j axes).

In [ ]:
# Create a batched DoubleZernikes (3 batch elements)
coefs_batch = rng.normal(0, 0.01, (3, 5, 15)) * u.um
dz_batch = DoubleZernikes(
    coefs=coefs_batch,
    field_outer=1.75 * u.deg, field_inner=0 * u.deg,
    pupil_outer=R_outer, pupil_inner=R_inner,
    frame="ocs", rtp=rtp,
)

print(f"batch_shape: {dz_batch.batch_shape}")
print(f"kmax: {dz_batch.kmax}, jmax: {dz_batch.jmax}")

dz0 = dz_batch[0]
print(f"dz_batch[0]: batch_shape={dz0.batch_shape}, kmax={dz0.kmax}")

## 8. Summary

| Property / Method | Description |
|---|---|
| `.coefs` | Coefficient array, shape `(..., kmax+1, jmax+1)` |
| `.kmax`, `.jmax`, `.eps` | Field and pupil properties |
| `.ocs`, `.ccs` | Frame conversion (rotates both field & pupil) |
| `.single(field)` | **Evaluate → Zernikes** at specific field points |
| `.to_galsim()` | Convert to `galsim.zernike.DoubleZernike` |
| `Zernikes.double(kmax, ...)` | **Fit from Zernikes** (reverse bridge) |

**Next:** [06_State.ipynb](06_State.ipynb) — alignment state representation
and basis conversions.